# Logistic Regression Baseline with Automated Dev Tuning & Logging

This notebook establishes the finalized, production-grade Logistic Regression baseline model. To ensure seamless structural integration across the group project, this pipeline fully mirrors the manual grid search, split-evaluation, and automated logging workflow used in our team's advanced scripts.

### Operational Workflow:
1. **Tuning Phase**: Fits a sequence of models over a specified hyperparameter grid exclusively on the `train` set and computes the optimization score (PR-AUC / Average Precision) exclusively on the `dev` set.
2. **Refit Phase**: Selects the optimal hyperparameter configuration, merges the `train` and `dev` partitions into a unified training matrix, fits a fresh `StandardScaler`, and retrains a comprehensive baseline model.
3. **Evaluation Phase**: Conducts a final test-set assessment, reporting all standardized project metrics in a clear text-based format.
4. **Logging Engine**: Dynamically creates a persistent `.log` file records every step, score, and final matrix configuration for group synchronization.

### Step 1: Environment Setup and Logging Configuration

In [5]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Any
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid

# Suppress standard training convergence warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TARGET_COLUMN = "class"

# Directory configurations aligning with group standards
PROJECT_DIR = Path('.').resolve()
DATA_DIR = PROJECT_DIR.parent / "outputs" / "processed_data"
LOGS_DIR = PROJECT_DIR / "outputs" / "logs"

def setup_outputs_and_logging() -> None:
    """Create output logs folder and configure the standard logging utility."""
    LOGS_DIR.mkdir(parents=True, exist_ok=True)
    
    # Flush existing handlers to avoid logging duplication in notebooks
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
        
    logging.basicConfig(
        filename=LOGS_DIR / "Baseline_LR_models.log",
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        filemode="w",
    )

def log_progress(message: str) -> None:
    """Simultaneously stream progress messages to standard output and log file."""
    print(message, flush=True)
    logging.info(message)

setup_outputs_and_logging()
log_progress("Initiating Logistic Regression Baseline Pipeline")

Initiating Logistic Regression Baseline Pipeline


### Step 2: Data Ingestion and Dimensionality Auditing

In [6]:
def split_features_target(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Isolate the predictor variables from the target column."""
    if TARGET_COLUMN not in data.columns:
        raise ValueError(f"Expected target column '{TARGET_COLUMN}' was not found.")
    x = data.drop(columns=[TARGET_COLUMN])
    y = data[TARGET_COLUMN].astype(int)
    return x, y

def load_data() -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """Ingest engineered train, dev, and test sets."""
    log_progress("Loading engineered train/dev/test data")
    
    train_df = pd.read_csv(DATA_DIR / "train_engineered.csv")
    dev_df = pd.read_csv(DATA_DIR / "dev_engineered.csv")
    test_df = pd.read_csv(DATA_DIR / "test_engineered.csv")

    x_train, y_train = split_features_target(train_df)
    x_dev, y_dev = split_features_target(dev_df)
    x_test, y_test = split_features_target(test_df)

    log_progress(f"Train shape: {x_train.shape}")
    log_progress(f"Dev shape: {x_dev.shape}")
    log_progress(f"Test shape: {x_test.shape}")
    return x_train, y_train, x_dev, y_dev, x_test, y_test

x_train, y_train, x_dev, y_dev, x_test, y_test = load_data()

Loading engineered train/dev/test data
Train shape: (12528, 12)
Dev shape: (2685, 12)
Test shape: (2685, 12)


### Step 3: Hyperparameter Optimization on the Development Set

In [7]:
def tune_with_dev_set(
    model_name: str,
    param_grid: dict[str, list[Any]],
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_dev: pd.DataFrame,
    y_dev: pd.Series,
) -> tuple[dict[str, Any], pd.DataFrame]:
    """Tune regularisation strengths by fitting on train and verifying on dev."""
    log_progress(f"Tuning {model_name} on the dev set")
    
    # Explicit scaling block restricted strictly to training parameters to prevent bleed
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_dev_scaled = scaler.transform(x_dev)

    rows = []
    best_score = -1.0
    best_params = None

    for params in ParameterGrid(param_grid):
        model = LogisticRegression(random_state=RANDOM_STATE, max_iter=2000, **params)
        model.fit(x_train_scaled, y_train)
        
        dev_proba = model.predict_proba(x_dev_scaled)[:, 1]
        dev_score = average_precision_score(y_dev, dev_proba)

        row = {
            "model": model_name,
            "dev_pr_auc_average_precision": dev_score,
            "params": params,
        }
        rows.append(row)
        log_progress(f"Evaluated Hyperparameters: {params} | Dev PR-AUC: {dev_score:.5f}")

        if dev_score > best_score:
            best_score = dev_score
            best_params = params

    tuning_results = pd.DataFrame(rows)
    tuning_results["rank_dev_pr_auc"] = (
        tuning_results["dev_pr_auc_average_precision"]
        .rank(ascending=False, method="min")
        .astype(int)
    )
    tuning_results = tuning_results.sort_values(by="rank_dev_pr_auc").reset_index(drop=True)

    log_progress(f"Best {model_name} parameters: {best_params}")
    log_progress(f"Best {model_name} dev PR-AUC: {best_score:.4f}")
    return best_params, tuning_results

lr_grid = {
    "C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

best_params, tuning_df = tune_with_dev_set(
    "Logistic Regression Baseline", lr_grid, x_train, y_train, x_dev, y_dev
)

Tuning Logistic Regression Baseline on the dev set
Evaluated Hyperparameters: {'C': 0.001} | Dev PR-AUC: 0.89782
Evaluated Hyperparameters: {'C': 0.01} | Dev PR-AUC: 0.91440
Evaluated Hyperparameters: {'C': 0.1} | Dev PR-AUC: 0.92292
Evaluated Hyperparameters: {'C': 1.0} | Dev PR-AUC: 0.91984
Evaluated Hyperparameters: {'C': 10.0} | Dev PR-AUC: 0.91706
Evaluated Hyperparameters: {'C': 100.0} | Dev PR-AUC: 0.91685
Best Logistic Regression Baseline parameters: {'C': 0.1}
Best Logistic Regression Baseline dev PR-AUC: 0.9229


### Step 4: Final Refit (Train+Dev) and Out-of-Sample Test Evaluation

In [8]:
def fit_final_model_and_evaluate(
    model_name: str,
    best_params: dict[str, Any],
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_dev: pd.DataFrame,
    y_dev: pd.Series,
    x_test: pd.DataFrame,
    y_test: pd.Series
) -> dict[str, Any]:
    """Merge train+dev partitions, fit a fresh scaler, and execute final test appraisal."""
    log_progress(f"Training final {model_name} on train + dev")
    
    # Merge partitions to expand information volume for the final estimator
    x_train_dev = pd.concat([x_train, x_dev], ignore_index=True)
    y_train_dev = pd.concat([y_train, y_dev], ignore_index=True)
    
    # Fit a comprehensive scaling standard across unified training subsets
    scaler = StandardScaler()
    x_train_dev_scaled = scaler.fit_transform(x_train_dev)
    x_test_scaled = scaler.transform(x_test)
    
    # Refit chosen model configuration
    model = LogisticRegression(random_state=RANDOM_STATE, max_iter=2000, **best_params)
    model.fit(x_train_dev_scaled, y_train_dev)
    
    log_progress(f"Evaluating final {model_name} on test set")
    y_pred = model.predict(x_test_scaled)
    y_proba = model.predict_proba(x_test_scaled)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc_average_precision": average_precision_score(y_test, y_proba),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }
    return metrics

final_metrics = fit_final_model_and_evaluate(
    "Logistic Regression Baseline", best_params, x_train, y_train, x_dev, y_dev, x_test, y_test
)

results_df = pd.DataFrame([final_metrics])
print("\n==================================================")
print("   FINAL MODEL EVALUATION (INDEPENDENT TEST SET)   ")
print("==================================================")
print(results_df.round(4).to_string(index=False))

log_progress("Logistic Regression Baseline training complete")
print(f"\nExecution log cleanly documented inside: {LOGS_DIR / 'Baseline_LR_models.log'}")

Training final Logistic Regression Baseline on train + dev
Evaluating final Logistic Regression Baseline on test set

   FINAL MODEL EVALUATION (INDEPENDENT TEST SET)   
                       model  accuracy  precision  recall  f1_score  roc_auc  pr_auc_average_precision   TN  FP  FN  TP
Logistic Regression Baseline    0.9765     0.9598  0.7764    0.8584   0.9703                    0.9131 2431   8  55 191
Logistic Regression Baseline training complete

Execution log cleanly documented inside: C:\Users\cdex1\Desktop\课件\49\ass2\数据\改2\outputs\logs\Baseline_LR_models.log
